# Lab 04_05 — CIFAR-10 Image Classification Using an ANN
**Solved implementation | Marks: 20**

## Objective
Build a configurable Artificial Neural Network (ANN) for CIFAR-10 image classification and investigate architecture, regularization, Batch Normalization, and optimizers.

### What this notebook covers
1. Load and preprocess CIFAR-10; visualize labelled examples.
2. Build a configurable fully connected ANN with adjustable hidden layers, neurons, activation, dropout, and Batch Normalization.
3. Compare **SGD, Adam, and AdamW**.
4. Compare baseline, L2/weight decay, Dropout, Batch Normalization, and combined regularization.
5. Plot training/validation loss and accuracy.
6. Evaluate the selected model with accuracy, precision, recall, F1, classification report, and confusion matrix.
7. Predict unseen test images and show confidence.
8. Visualize errors and frequent class confusions.
9. Run a small hyperparameter study.
10. Select a model using validation performance and summarize its test results.

> **Important:** Results are generated when you run the notebook. No experimental scores are hard-coded. The default settings are intended to make the experiments manageable on a typical Colab/runtime; increase the settings for a more thorough study.

## 1. Imports and reproducibility
The model is a **dense ANN**, not a CNN. Each 32×32 RGB image is flattened into a vector of 3,072 values. This makes it possible to study ANN architecture and regularization directly, though spatial image structure is not explicitly preserved.

In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Available devices:", tf.config.list_physical_devices())

## 2. Experiment settings
To run a faster demonstration, reduce `TRAIN_LIMIT`, `EPOCHS`, or the number of configurations. For a stronger final submission, use all 50,000 training images and more epochs if compute permits.

- `TRAIN_LIMIT = None` uses all training examples.
- `TEST_LIMIT = None` uses all 10,000 test examples.
- The official CIFAR-10 test set is held out from fitting; model selection uses validation accuracy.

In [ ]:
# Adjust these settings to suit your runtime.
TRAIN_LIMIT = 20000       # Set to None to use all 50,000 training examples
TEST_LIMIT = None         # Set to an integer for a quicker test, or None for all 10,000
EPOCHS = 5
BATCH_SIZE = 128
VERBOSE = 1

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

## 3. Load and preprocess CIFAR-10
Pixel values are converted from integers in `[0, 255]` to floating-point values in `[0, 1]`. Labels are kept as integer class IDs because the model uses sparse categorical cross-entropy.

In [ ]:
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = tf.keras.datasets.cifar10.load_data()

y_train_raw = y_train_raw.squeeze()
y_test_raw = y_test_raw.squeeze()

if TRAIN_LIMIT is not None:
    x_train_raw = x_train_raw[:TRAIN_LIMIT]
    y_train_raw = y_train_raw[:TRAIN_LIMIT]

if TEST_LIMIT is not None:
    x_test_raw = x_test_raw[:TEST_LIMIT]
    y_test_raw = y_test_raw[:TEST_LIMIT]

x_train = x_train_raw.astype("float32") / 255.0
x_test = x_test_raw.astype("float32") / 255.0

# Flatten 32x32x3 images for a fully connected ANN.
x_train_flat = x_train.reshape(len(x_train), -1)
x_test_flat = x_test.reshape(len(x_test), -1)

print("Training images:", x_train.shape, "| Flattened:", x_train_flat.shape)
print("Test images:", x_test.shape, "| Flattened:", x_test_flat.shape)
print("Number of classes:", len(np.unique(y_train_raw)))

In [ ]:
# Display representative images with their class labels.
plt.figure(figsize=(12, 7))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(CLASS_NAMES[int(y_train_raw[i])])
    plt.axis("off")
plt.suptitle("Representative CIFAR-10 training images", y=1.02)
plt.tight_layout()
plt.show()

class_counts = pd.Series(y_train_raw).value_counts().sort_index()
plt.figure(figsize=(9, 4))
plt.bar(CLASS_NAMES, [class_counts.get(i, 0) for i in range(10)])
plt.title("Training examples per class")
plt.ylabel("Number of images")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 4. Configurable ANN builder
The builder exposes:
- `hidden_layers`: number of hidden layers and neurons in each layer.
- `activation`: activation function for hidden layers.
- `dropout_rate`: dropout probability.
- `batch_norm`: whether to add Batch Normalization.
- `l2_strength`: L2 kernel regularization strength.
- `optimizer_name` and `learning_rate`: optimizer configuration.

The output layer has 10 units and softmax activation, producing a probability distribution across CIFAR-10 classes.

In [ ]:
def make_optimizer(name="adam", learning_rate=1e-3):
    name = name.lower()
    if name == "sgd":
        return tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    if name == "adam":
        return tf.keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "adamw":
        # AdamW applies decoupled weight decay.
        return tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=1e-4
        )
    raise ValueError("optimizer_name must be one of: 'sgd', 'adam', 'adamw'")


def build_ann(
    input_dim=3072,
    hidden_layers=(512, 256),
    activation="relu",
    dropout_rate=0.0,
    batch_norm=False,
    l2_strength=0.0,
    optimizer_name="adam",
    learning_rate=1e-3
):
    regularizer = tf.keras.regularizers.l2(l2_strength) if l2_strength > 0 else None

    model = tf.keras.Sequential(name="CIFAR10_FullyConnected_ANN")
    model.add(tf.keras.layers.Input(shape=(input_dim,)))

    for units in hidden_layers:
        model.add(tf.keras.layers.Dense(
            units,
            activation=None if batch_norm else activation,
            kernel_regularizer=regularizer
        ))
        if batch_norm:
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Activation(activation))
        if dropout_rate > 0:
            model.add(tf.keras.layers.Dropout(dropout_rate))

    model.add(tf.keras.layers.Dense(10, activation="softmax"))
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


# Build and inspect one example model (not trained yet).
example_model = build_ann(
    hidden_layers=(256, 128),
    activation="relu",
    dropout_rate=0.25,
    batch_norm=True,
    optimizer_name="adam",
    learning_rate=1e-3
)
example_model.summary()

## 5. Helper function for training experiments
We use the same train/validation split and evaluation approach for each configuration. Validation accuracy is used for model selection; the test set is reserved for final evaluation.

In [ ]:
# Fixed validation set for fairer comparisons across configurations.
# Use the last 10% of the selected training data as validation.
split_at = int(len(x_train_flat) * 0.9)
x_fit, x_val = x_train_flat[:split_at], x_train_flat[split_at:]
y_fit, y_val = y_train_raw[:split_at], y_train_raw[split_at:]

print("Fit set:", x_fit.shape, "Validation set:", x_val.shape)


def run_experiment(config, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=VERBOSE):
    tf.keras.backend.clear_session()
    model = build_ann(
        hidden_layers=config.get("hidden_layers", (512, 256)),
        activation=config.get("activation", "relu"),
        dropout_rate=config.get("dropout_rate", 0.0),
        batch_norm=config.get("batch_norm", False),
        l2_strength=config.get("l2_strength", 0.0),
        optimizer_name=config.get("optimizer_name", "adam"),
        learning_rate=config.get("learning_rate", 1e-3)
    )
    start = time.time()
    history = model.fit(
        x_fit, y_fit,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose
    )
    elapsed = time.time() - start
    best_epoch = int(np.argmax(history.history["val_accuracy"])) + 1
    best_val_accuracy = float(np.max(history.history["val_accuracy"]))
    return {
        "name": config["name"],
        "config": config.copy(),
        "model": model,
        "history": history.history,
        "seconds": elapsed,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_val_accuracy,
        "batch_size": batch_size,
        "epochs": epochs
    }


def plot_history(result, title=None):
    hist = result["history"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist["loss"], label="Training loss")
    axes[0].plot(hist["val_loss"], label="Validation loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(hist["accuracy"], label="Training accuracy")
    axes[1].plot(hist["val_accuracy"], label="Validation accuracy")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    fig.suptitle(title or result["name"])
    plt.tight_layout()
    plt.show()

## 6. Optimizer comparison: SGD vs Adam vs AdamW
This first experiment keeps the architecture and regularization settings the same, changing only the optimizer. This makes the comparison more interpretable.

In [ ]:
optimizer_configs = [
    {
        "name": "Optimizer — SGD",
        "optimizer_name": "sgd",
        "learning_rate": 0.01,
        "hidden_layers": (512, 256),
        "activation": "relu",
        "dropout_rate": 0.0,
        "batch_norm": False,
        "l2_strength": 0.0
    },
    {
        "name": "Optimizer — Adam",
        "optimizer_name": "adam",
        "learning_rate": 1e-3,
        "hidden_layers": (512, 256),
        "activation": "relu",
        "dropout_rate": 0.0,
        "batch_norm": False,
        "l2_strength": 0.0
    },
    {
        "name": "Optimizer — AdamW",
        "optimizer_name": "adamw",
        "learning_rate": 1e-3,
        "hidden_layers": (512, 256),
        "activation": "relu",
        "dropout_rate": 0.0,
        "batch_norm": False,
        "l2_strength": 0.0
    }
]

optimizer_results = []
for cfg in optimizer_configs:
    print("\nTraining:", cfg["name"])
    result = run_experiment(cfg)
    optimizer_results.append(result)
    print(f"Best validation accuracy: {result['best_val_accuracy']:.4f} at epoch {result['best_epoch']} "
          f"({result['seconds']:.1f}s)")

In [ ]:
for result in optimizer_results:
    plot_history(result)

optimizer_summary = pd.DataFrame([
    {
        "Experiment": r["name"],
        "Optimizer": r["config"]["optimizer_name"],
        "Learning rate": r["config"]["learning_rate"],
        "Best validation accuracy": r["best_val_accuracy"],
        "Best epoch": r["best_epoch"],
        "Training time (s)": r["seconds"]
    }
    for r in optimizer_results
]).sort_values("Best validation accuracy", ascending=False)

display(optimizer_summary)

## 7. Regularization comparison
The following experiments compare:
1. **Baseline** — no explicit regularization.
2. **L2 regularization** — penalizes large weights.
3. **Dropout** — randomly disables a fraction of hidden activations during training.
4. **Batch Normalization** — normalizes hidden activations within mini-batches.
5. **Combined** — L2 + Dropout + Batch Normalization.

Adam is used in all five experiments to keep the optimizer fixed.

In [ ]:
regularization_configs = [
    {
        "name": "Regularization — Baseline",
        "optimizer_name": "adam", "learning_rate": 1e-3,
        "hidden_layers": (512, 256), "activation": "relu",
        "dropout_rate": 0.0, "batch_norm": False, "l2_strength": 0.0
    },
    {
        "name": "Regularization — L2",
        "optimizer_name": "adam", "learning_rate": 1e-3,
        "hidden_layers": (512, 256), "activation": "relu",
        "dropout_rate": 0.0, "batch_norm": False, "l2_strength": 1e-4
    },
    {
        "name": "Regularization — Dropout",
        "optimizer_name": "adam", "learning_rate": 1e-3,
        "hidden_layers": (512, 256), "activation": "relu",
        "dropout_rate": 0.30, "batch_norm": False, "l2_strength": 0.0
    },
    {
        "name": "Regularization — Batch Normalization",
        "optimizer_name": "adam", "learning_rate": 1e-3,
        "hidden_layers": (512, 256), "activation": "relu",
        "dropout_rate": 0.0, "batch_norm": True, "l2_strength": 0.0
    },
    {
        "name": "Regularization — Combined",
        "optimizer_name": "adam", "learning_rate": 1e-3,
        "hidden_layers": (512, 256), "activation": "relu",
        "dropout_rate": 0.30, "batch_norm": True, "l2_strength": 1e-4
    }
]

regularization_results = []
for cfg in regularization_configs:
    print("\nTraining:", cfg["name"])
    result = run_experiment(cfg)
    regularization_results.append(result)
    print(f"Best validation accuracy: {result['best_val_accuracy']:.4f} at epoch {result['best_epoch']} "
          f"({result['seconds']:.1f}s)")

In [ ]:
for result in regularization_results:
    plot_history(result)

regularization_summary = pd.DataFrame([
    {
        "Experiment": r["name"],
        "Dropout": r["config"]["dropout_rate"],
        "Batch Normalization": r["config"]["batch_norm"],
        "L2 strength": r["config"]["l2_strength"],
        "Best validation accuracy": r["best_val_accuracy"],
        "Best epoch": r["best_epoch"],
        "Training time (s)": r["seconds"]
    }
    for r in regularization_results
]).sort_values("Best validation accuracy", ascending=False)

display(regularization_summary)

## 8. Small hyperparameter study
We vary the hidden-layer architecture, learning rate, and batch size. These are short experiments, so use their results as a small comparison rather than proof of a globally optimal configuration.

In [ ]:
hyperparameter_configs = [
    {
        "name": "Hyperparameter — Small network",
        "optimizer_name": "adam", "learning_rate": 1e-3,
        "hidden_layers": (256, 128), "activation": "relu",
        "dropout_rate": 0.25, "batch_norm": True, "l2_strength": 1e-4,
        "_batch_size": 128
    },
    {
        "name": "Hyperparameter — Lower learning rate",
        "optimizer_name": "adam", "learning_rate": 3e-4,
        "hidden_layers": (512, 256), "activation": "relu",
        "dropout_rate": 0.30, "batch_norm": True, "l2_strength": 1e-4,
        "_batch_size": 128
    },
    {
        "name": "Hyperparameter — Larger batch",
        "optimizer_name": "adam", "learning_rate": 1e-3,
        "hidden_layers": (512, 256), "activation": "relu",
        "dropout_rate": 0.30, "batch_norm": True, "l2_strength": 1e-4,
        "_batch_size": 256
    }
]

hyperparameter_results = []
for cfg in hyperparameter_configs:
    cfg_for_model = {k: v for k, v in cfg.items() if not k.startswith("_")}
    batch_size = cfg.get("_batch_size", BATCH_SIZE)
    print("\nTraining:", cfg["name"], "| batch size:", batch_size)
    result = run_experiment(cfg_for_model, batch_size=batch_size)
    hyperparameter_results.append(result)
    print(f"Best validation accuracy: {result['best_val_accuracy']:.4f} at epoch {result['best_epoch']}")

In [ ]:
for result in hyperparameter_results:
    plot_history(result)

hyperparameter_summary = pd.DataFrame([
    {
        "Experiment": r["name"],
        "Architecture": str(r["config"]["hidden_layers"]),
        "Learning rate": r["config"]["learning_rate"],
        "Batch size": r["batch_size"],
        "Dropout": r["config"]["dropout_rate"],
        "Batch Normalization": r["config"]["batch_norm"],
        "L2 strength": r["config"]["l2_strength"],
        "Best validation accuracy": r["best_val_accuracy"]
    }
    for r in hyperparameter_results
]).sort_values("Best validation accuracy", ascending=False)

display(hyperparameter_summary)

## 9. Select the best configuration
The selected model is the run with the highest **validation accuracy** across all experiments. The test set is not used for selection. If two configurations are very close, consider preferring the simpler model or the one with more stable validation curves.

In [ ]:
all_results = optimizer_results + regularization_results + hyperparameter_results
best_result = max(all_results, key=lambda r: r["best_val_accuracy"])
best_model = best_result["model"]

print("Selected experiment:", best_result["name"])
print("Configuration:", best_result["config"])
print(f"Best validation accuracy: {best_result['best_val_accuracy']:.4f}")
print("Best epoch:", best_result["best_epoch"])
print("Note: this selects the best trained run; it does not retrain on the validation set.")

## 10. Final test evaluation
Report accuracy, macro precision, macro recall, macro F1, the per-class classification report, and a confusion matrix. Macro averages give each class equal weight.

In [ ]:
test_probabilities = best_model.predict(x_test_flat, batch_size=BATCH_SIZE, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)

test_accuracy = accuracy_score(y_test_raw, test_predictions)
test_precision = precision_score(y_test_raw, test_predictions, average="macro", zero_division=0)
test_recall = recall_score(y_test_raw, test_predictions, average="macro", zero_division=0)
test_f1 = f1_score(y_test_raw, test_predictions, average="macro", zero_division=0)

print("FINAL TEST METRICS")
print(f"Accuracy        : {test_accuracy:.4f}")
print(f"Macro precision : {test_precision:.4f}")
print(f"Macro recall    : {test_recall:.4f}")
print(f"Macro F1-score  : {test_f1:.4f}")
print("\nClassification report:")
print(classification_report(
    y_test_raw, test_predictions, target_names=CLASS_NAMES, zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test_raw, test_predictions)
fig, ax = plt.subplots(figsize=(10, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("Confusion matrix — selected ANN")
plt.tight_layout()
plt.show()

## 11. Predictions on unseen test images
For each image, the model returns ten class probabilities. The predicted class is the class with the highest probability; confidence is that maximum probability. These are test images the model did not use during fitting.

In [ ]:
rng = np.random.default_rng(SEED)
sample_count = min(15, len(x_test))
sample_indices = rng.choice(len(x_test), size=sample_count, replace=False)

sample_probs = test_probabilities[sample_indices]
sample_preds = np.argmax(sample_probs, axis=1)
sample_conf = np.max(sample_probs, axis=1)

plt.figure(figsize=(14, 9))
for j, idx in enumerate(sample_indices):
    actual = int(y_test_raw[idx])
    pred = int(sample_preds[j])
    plt.subplot(3, 5, j + 1)
    plt.imshow(x_test[idx])
    plt.title(
        f"Actual: {CLASS_NAMES[actual]}\nPred: {CLASS_NAMES[pred]} ({sample_conf[j]:.1%})",
        color=("green" if actual == pred else "red"),
        fontsize=9
    )
    plt.axis("off")
plt.suptitle("Predictions on unseen CIFAR-10 test images (green = correct, red = incorrect)")
plt.tight_layout()
plt.show()

prediction_table = pd.DataFrame({
    "Test index": sample_indices,
    "Actual class": [CLASS_NAMES[int(y_test_raw[i])] for i in sample_indices],
    "Predicted class": [CLASS_NAMES[int(p)] for p in sample_preds],
    "Confidence": sample_conf,
    "Correct?": [bool(y_test_raw[i] == p) for i, p in zip(sample_indices, sample_preds)]
})
display(prediction_table)

## 12. Incorrectly classified images and common confusions
Misclassifications help show which categories the ANN struggles to distinguish. CIFAR-10 images are small and often contain similar shapes, textures, or backgrounds, so some mistakes are understandable.

In [ ]:
wrong_indices = np.where(test_predictions != y_test_raw)[0]
print(f"Incorrectly classified: {len(wrong_indices)} / {len(y_test_raw)} "
      f"({len(wrong_indices) / len(y_test_raw):.1%})")

show_n = min(20, len(wrong_indices))
if show_n > 0:
    chosen_wrong = wrong_indices[:show_n]
    plt.figure(figsize=(14, 9))
    for j, idx in enumerate(chosen_wrong):
        actual = int(y_test_raw[idx])
        pred = int(test_predictions[idx])
        confidence = float(test_probabilities[idx, pred])
        plt.subplot(4, 5, j + 1)
        plt.imshow(x_test[idx])
        plt.title(
            f"True: {CLASS_NAMES[actual]}\nPred: {CLASS_NAMES[pred]} ({confidence:.1%})",
            fontsize=8
        )
        plt.axis("off")
    plt.suptitle("Examples of incorrect predictions")
    plt.tight_layout()
    plt.show()
else:
    print("No errors found in this test subset.")

In [ ]:
# Find the most frequent off-diagonal confusion pairs.
cm_offdiag = cm.copy()
np.fill_diagonal(cm_offdiag, 0)

pairs = []
for true_id in range(10):
    for pred_id in range(10):
        if true_id != pred_id and cm_offdiag[true_id, pred_id] > 0:
            pairs.append({
                "True class": CLASS_NAMES[true_id],
                "Predicted class": CLASS_NAMES[pred_id],
                "Count": int(cm_offdiag[true_id, pred_id])
            })

confusion_pairs = pd.DataFrame(pairs)
if not confusion_pairs.empty:
    confusion_pairs = confusion_pairs.sort_values("Count", ascending=False).head(10)
    print("Most frequent class confusions:")
    display(confusion_pairs)
else:
    print("No off-diagonal confusions were observed.")

## 13. Discussion questions

### Q1. Which optimizer and regularization technique performed best? Did Batch Normalization and Dropout reduce overfitting?
Use the tables and learning curves above to answer this from the actual run. The following cell prints a short evidence-based summary using the measured validation results. A technique should be described as helping when it improves validation performance and/or narrows the training–validation gap; it is not guaranteed to win on every run.

### Q2. Which classes were most frequently misclassified, and why might an ANN be less suitable for image classification than a CNN?
The confusion-pair table identifies the most frequent class mix-ups for this run. A dense ANN flattens each image, so pixel positions are treated as separate input features. It does not naturally encode local spatial patterns or translation invariance. A CNN uses local filters and shared weights to learn edges, textures, and higher-level spatial features more efficiently, often making it better suited to images.

In [ ]:
# Evidence-based summary for the discussion section.
optimizer_winner = optimizer_summary.iloc[0]
regularization_winner = regularization_summary.iloc[0]

print("DISCUSSION SUMMARY — based on this notebook's results")
print(
    f"1) Among the tested optimizer runs, {optimizer_winner['Optimizer']} had the highest "
    f"validation accuracy ({optimizer_winner['Best validation accuracy']:.4f})."
)
print(
    f"   Among the regularization experiments, '{regularization_winner['Experiment']}' had the "
    f"highest validation accuracy ({regularization_winner['Best validation accuracy']:.4f})."
)

baseline_result = next(r for r in regularization_results if r["name"] == "Regularization — Baseline")
combined_result = next(r for r in regularization_results if r["name"] == "Regularization — Combined")

def train_val_gap(result):
    h = result["history"]
    return float(h["accuracy"][-1] - h["val_accuracy"][-1])

print(
    f"   Final-epoch train–validation accuracy gap: baseline={train_val_gap(baseline_result):.4f}, "
    f"combined={train_val_gap(combined_result):.4f}."
)
if train_val_gap(combined_result) < train_val_gap(baseline_result):
    print("   In this run, the combined regularization model had a smaller train–validation gap than baseline.")
else:
    print("   In this run, combined regularization did not reduce the final-epoch accuracy gap versus baseline; "
          "regularization effects depend on the data split and hyperparameters.")

if not confusion_pairs.empty:
    top_pair = confusion_pairs.iloc[0]
    print(
        f"2) Most frequent specific confusion in this run: true '{top_pair['True class']}' predicted as "
        f"'{top_pair['Predicted class']}' ({int(top_pair['Count'])} examples)."
    )
else:
    print("2) No off-diagonal class confusions were recorded.")
print(
    "   General interpretation: a flattened ANN lacks the local connectivity and weight sharing of CNNs, "
    "so it may need more parameters and may learn image structure less efficiently."
)

## 14. Final report summary
This cell consolidates the selected architecture, optimizer, hyperparameters, and measured test performance for your report. Copy the generated values into your written conclusion if your instructor requires a separate report.

In [ ]:
final_config = best_result["config"]
final_summary = pd.DataFrame([{
    "Selected experiment": best_result["name"],
    "Architecture": "Input 3072 → " + " → ".join(map(str, final_config.get("hidden_layers", (512, 256)))) + " → Output 10",
    "Activation": final_config.get("activation", "relu"),
    "Optimizer": final_config.get("optimizer_name", "adam"),
    "Learning rate": final_config.get("learning_rate", 1e-3),
    "Dropout rate": final_config.get("dropout_rate", 0.0),
    "Batch normalization": final_config.get("batch_norm", False),
    "L2 strength": final_config.get("l2_strength", 0.0),
    "Batch size": best_result["batch_size"],
    "Epochs run": best_result["epochs"],
    "Best validation accuracy": best_result["best_val_accuracy"],
    "Test accuracy": test_accuracy,
    "Macro precision": test_precision,
    "Macro recall": test_recall,
    "Macro F1": test_f1
}])
display(final_summary.T.rename(columns={0: "Final result"}))

print(
    "\nConclusion template:\n"
    f"The selected ANN configuration was '{best_result['name']}', with hidden layers "
    f"{final_config.get('hidden_layers', (512, 256))}, optimizer "
    f"{final_config.get('optimizer_name', 'adam')}, and learning rate "
    f"{final_config.get('learning_rate', 1e-3)}. It achieved validation accuracy "
    f"{best_result['best_val_accuracy']:.4f} and test accuracy {test_accuracy:.4f}, "
    f"with macro precision {test_precision:.4f}, macro recall {test_recall:.4f}, "
    f"and macro F1 {test_f1:.4f}. The confusion matrix and misclassified examples above "
    "show which classes were difficult for this dense ANN. Results may change with training "
    "subset, random seed, runtime, and hyperparameters."
)

---
### Notes for submission
- Run **all cells from top to bottom** so every table and plot is generated.
- If your runtime is slow, lower `TRAIN_LIMIT` and `EPOCHS` first. For a more reliable final experiment, increase them and rerun.
- Keep the actual outputs from your own run; do not report guessed metrics.
- This notebook uses a dense ANN as requested. A CNN comparison could be an optional extension, but it is not substituted for the required ANN experiments.